In [300]:
import kagglehub
import pandas as pd 
import numpy as np 
from sklearn.preprocessing import MinMaxScaler


## Loading Data

In [301]:
# # Download latest version
# path = kagglehub.dataset_download("arindam235/startup-investments-crunchbase")

# print("Path to dataset files:", path)

In [302]:
data = pd.read_csv('/Users/vivianjohnson/.cache/kagglehub/datasets/arindam235/startup-investments-crunchbase/versions/1/investments_VC.csv',
                   encoding='ISO-8859-1')

## Data Cleaning

In [303]:
data.describe().T

,count,mean,std,min,25%,50%,75%,max
funding_rounds,49438.0,1.696205e+00,1.294213e+00,1.0,1.0,1.0,2.0,1.800000e+01
founded_year,38482.0,2.007359e+03,7.579203e+00,1902.0,2006.0,2010.0,2012.0,2.014000e+03
seed,49438.0,2.173215e+05,1.056985e+06,0.0,0.0,0.0,25000.0,1.300000e+08
venture,49438.0,7.501051e+06,2.847112e+07,0.0,0.0,0.0,5000000.0,2.351000e+09
equity_crowdfunding,49438.0,6.163322e+03,1.999048e+05,0.0,0.0,0.0,0.0,2.500000e+07
undisclosed,49438.0,1.302213e+05,2.981404e+06,0.0,0.0,0.0,0.0,2.924328e+08
convertible_note,49438.0,2.336410e+04,1.432046e+06,0.0,0.0,0.0,0.0,3.000000e+08
debt_financing,49438.0,1.888157e+06,1.382046e+08,0.0,0.0,0.0,0.0,3.007950e+10
angel,49438.0,6.541898e+04,6.582908e+05,0.0,0.0,0.0,0.0,6.359026e+07
grant,49438.0,1.628453e+05,5.612088e+06,0.0,0.0,0.0,0.0,7.505000e+08


In [304]:
print(f'Initial len: {len(data)}')

Initial len: 54294


The length of the data is 54294 but many null

In [305]:
# remove obvs where name is null 
data = data[~data.name.isna()]
print(f'Len after remove null names: {len(data)}')

Len after remove null names: 49437


In [306]:
# list of columns
data.columns

Index(['permalink', 'name', 'homepage_url', 'category_list', ' market ',
       ' funding_total_usd ', 'status', 'country_code', 'state_code', 'region',
       'city', 'funding_rounds', 'founded_at', 'founded_month',
       'founded_quarter', 'founded_year', 'first_funding_at',
       'last_funding_at', 'seed', 'venture', 'equity_crowdfunding',
       'undisclosed', 'convertible_note', 'debt_financing', 'angel', 'grant',
       'private_equity', 'post_ipo_equity', 'post_ipo_debt',
       'secondary_market', 'product_crowdfunding', 'round_A', 'round_B',
       'round_C', 'round_D', 'round_E', 'round_F', 'round_G', 'round_H'],
      dtype='object')

In [307]:
# remove spaces in col names 
data.rename(columns={' funding_total_usd ' : 'funding_total_usd',
                     ' market ' : 'market'},
            inplace=True)

data.columns

Index(['permalink', 'name', 'homepage_url', 'category_list', 'market',
       'funding_total_usd', 'status', 'country_code', 'state_code', 'region',
       'city', 'funding_rounds', 'founded_at', 'founded_month',
       'founded_quarter', 'founded_year', 'first_funding_at',
       'last_funding_at', 'seed', 'venture', 'equity_crowdfunding',
       'undisclosed', 'convertible_note', 'debt_financing', 'angel', 'grant',
       'private_equity', 'post_ipo_equity', 'post_ipo_debt',
       'secondary_market', 'product_crowdfunding', 'round_A', 'round_B',
       'round_C', 'round_D', 'round_E', 'round_F', 'round_G', 'round_H'],
      dtype='object')

Useful columns: 

identity cols 
- name 
- category_list / market [industry filtering]
- country_code / state_code / region / city [geographical analysis]

funding metrics
- funding_total_usd
- funding_rounds 
- seed / venture / round_A / round_B / round_C / round_D / ... [stages of funding]
- first_funding_at / last_funding_at [funding timeline length]

outcome / timeline 
- status [outcome]
- founded_at [date]

In [308]:
# filter cols to keep 
COLS_TO_KEEP = ['name','category_list','market',
                'funding_total_usd','status','country_code',
                'state_code','city','funding_rounds',
                'founded_at','first_funding_at','last_funding_at','seed',
                'venture','round_A','round_B',
                'round_C','round_D']

data = data[COLS_TO_KEEP]

In [309]:
# check num nulls 
data.isnull().sum()

name                     0
category_list         3961
market                3968
funding_total_usd        0
status                1314
country_code          5272
state_code           19276
city                  6115
funding_rounds           0
founded_at           10884
first_funding_at         0
last_funding_at          0
seed                     0
venture                  0
round_A                  0
round_B                  0
round_C                  0
round_D                  0
dtype: int64

founded_at can't be null, need to calculate age - drop rows 
status is outcome varible, can't score without knowing if success - drop rows 

fill market, category_list, city with 'unknown'

filter to US only 

In [310]:
# filter to us only 
data = data[data['country_code'] == 'USA']

# drop country code
data = data.drop(columns=['country_code', 'state_code'])

# drop null status 
data = data[~data.status.isnull()]

# drop null founded at  
data = data[~data.founded_at.isnull()]

# fill missing category, market, city with unknown 
data['category_list'] = data['category_list'].fillna('Unknown')
data['market'] = data['market'].fillna('Unknown')
data['city'] = data['city'].fillna('Unknown')

print(data.isnull().sum())
print(f"\nDataset shape: {data.shape}")

name                 0
category_list        0
market               0
funding_total_usd    0
status               0
city                 0
funding_rounds       0
founded_at           0
first_funding_at     0
last_funding_at      0
seed                 0
venture              0
round_A              0
round_B              0
round_C              0
round_D              0
dtype: int64

Dataset shape: (23243, 16)


In [311]:
# turn cateogry_list into comma sep array 
data['category_list'] = data['category_list'].astype(str)

In [312]:
# convert funding to numeric - fill missing w nas
data['funding_total_usd'] = data['funding_total_usd'].str.replace(r'[^\d.]', '', regex=True)
data['funding_total_usd'] = pd.to_numeric(data['funding_total_usd'], errors='coerce').fillna(0.0)

In [313]:
data.head()

,name,category_list,market,funding_total_usd,status,city,funding_rounds,founded_at,first_funding_at,last_funding_at,seed,venture,round_A,round_B,round_C,round_D
0,#waywire,|Entertainment|Politics|Social Media|News|,News,1750000.0,acquired,New York,1.0,2012-06-01,2012-06-30,2012-06-30,1750000.0,0.0,0.0,0.0,0.0,0.0
4,-R- Ranch and Mine,|Tourism|Entertainment|Games|,Tourism,60000.0,operating,Fort Worth,2.0,2014-01-01,2014-08-17,2014-09-26,0.0,0.0,0.0,0.0,0.0,0.0
8,004 Technologies,|Software|,Software,0.0,operating,Champaign,1.0,2010-01-01,2014-07-24,2014-07-24,0.0,0.0,0.0,0.0,0.0,0.0
12,1-800-DENTIST,|Health and Wellness|,Health and Wellness,0.0,operating,Los Angeles,1.0,1986-01-01,2010-08-19,2010-08-19,0.0,0.0,0.0,0.0,0.0,0.0
13,1-800-DOCTORS,|Health and Wellness|,Health and Wellness,1750000.0,operating,Iselin,1.0,1984-01-01,2011-03-02,2011-03-02,0.0,0.0,0.0,0.0,0.0,0.0


## Feature Engineering

### funding per round 

total funding / number of rounds 

lower = more efficient

In [314]:
data['funding_per_round'] = data['funding_total_usd'] / data['funding_rounds']

data['funding_per_round_clean'] = data['funding_per_round'].replace(0, data['funding_per_round'].median())

# invert because lower is better - want to reflect when adding together 
data['inv_funding_per_round'] = 1 / data['funding_per_round_clean']

### speed to first funding 

how quickly after founding did they attract investors 
faster = more attractive business

In [315]:
data['founded_at'] = pd.to_datetime(data['founded_at'],
                                    format='%Y-%m-%d',
                                    errors='coerce')
data['first_funding_at'] = pd.to_datetime(data['first_funding_at'],
                                          format='%Y-%m-%d',
                                          errors='coerce')
data['last_funding_at'] = pd.to_datetime(data['last_funding_at'],
                                          format='%Y-%m-%d',
                                          errors='coerce')

data['days_to_first_funding'] = (data['first_funding_at'] - data['founded_at']).dt.days

data['days_to_first_funding'] = data['days_to_first_funding'].clip(lower=0)
data['inv_days_to_funding'] = 1 / (data['days_to_first_funding'] + 1)

### outcome score 

score of the status col 

In [316]:
outcome_map = {
    'acquired' : 3,
    'operating' : 2,
    'closed' : 0
}

data['outcome_score'] = data['status'].map(outcome_map)

### funding window days

In [317]:
data['funding_window_days'] = (data['last_funding_at'] - data['first_funding_at']).dt.days

### efficiency score 

combine normalized vars into a single score from 0-100 

outcome is most important - want to penalize those that failed 

capital efficiency = 25% 

speed to funding = 20% 

funding window = 15% 

In [318]:
SCORE_FEATURES = ['outcome_score', 'inv_funding_per_round', 
                  'inv_days_to_funding', 'funding_window_days']

# normalize 
scaler = MinMaxScaler()
data[SCORE_FEATURES] = scaler.fit_transform(data[SCORE_FEATURES])

In [319]:
# weighted final efficiency score 
data['efficiency_score'] = (
    data['outcome_score'] * 0.40 +
    data['inv_funding_per_round'] * 0.25 +
    data['inv_days_to_funding'] * 0.20 +
    data['funding_window_days'] * 0.15
) * 100

In [320]:
data.head()

,name,category_list,market,funding_total_usd,status,city,funding_rounds,founded_at,first_funding_at,last_funding_at,...,round_C,round_D,funding_per_round,funding_per_round_clean,inv_funding_per_round,days_to_first_funding,inv_days_to_funding,outcome_score,funding_window_days,efficiency_score
0,#waywire,|Entertainment|Politics|Social Media|News|,News,1750000.0,acquired,New York,1.0,2012-06-01,2012-06-30,2012-06-30,...,0.0,0.0,1750000.0,1.750000e+06,0.000017,29.0,0.033321,1.000000,0.000000,40.666839
4,-R- Ranch and Mine,|Tourism|Entertainment|Games|,Tourism,60000.0,operating,Fort Worth,2.0,2014-01-01,2014-08-17,2014-09-26,...,0.0,0.0,30000.0,3.000000e+04,0.001000,228.0,0.004354,0.666667,0.003979,26.838429
8,004 Technologies,|Software|,Software,0.0,operating,Champaign,1.0,2010-01-01,2014-07-24,2014-07-24,...,0.0,0.0,0.0,1.016667e+06,0.000030,1665.0,0.000587,0.666667,0.000000,26.679145
12,1-800-DENTIST,|Health and Wellness|,Health and Wellness,0.0,operating,Los Angeles,1.0,1986-01-01,2010-08-19,2010-08-19,...,0.0,0.0,0.0,1.016667e+06,0.000030,8996.0,0.000098,0.666667,0.000000,26.669363
13,1-800-DOCTORS,|Health and Wellness|,Health and Wellness,1750000.0,operating,Iselin,1.0,1984-01-01,2011-03-02,2011-03-02,...,0.0,0.0,1750000.0,1.750000e+06,0.000017,9922.0,0.000088,0.666667,0.000000,26.668846


## SQL Analysis 

In [322]:
import sqlite3

conn = sqlite3.connect('startups.db')
data.to_sql('startups',
            conn,
            if_exists='replace',
            index=False)

print('Database created successfully')
print(f"Rows loaded: {pd.read_sql_query('SELECT COUNT(*) as count FROM startups', conn).iloc[0,0]}")

Database created successfully
Rows loaded: 23243


In [324]:
# helper function to run query 
def run_query(query):
    return pd.read_sql_query(query, conn)

### Efficiency Score Leaderboard 

Which startups have the highest efficiency score? 

In [335]:
query1 = """
    SELECT 
        name, 
        market, 
        city, 
        status,
        ROUND(efficiency_score, 1) as efficiency_score, 
        ROUND(funding_total_usd, 0) as total_funding
    FROM startups
    ORDER BY efficiency_score DESC
"""
q1 = run_query(query=query1)
q1.to_csv('leaderboard.csv', index=False)

### Industry Breakdown

Which industries produce the most efficient startups on aveage 

In [336]:
query2 = """
SELECT
    market,
    COUNT(*) as company_count,
    ROUND(AVG(efficiency_score), 2) as avg_efficiency_score,
    ROUND(AVG(funding_total_usd), 2) as avg_funding,
    SUM(CASE WHEN status = 'acquired' THEN 1 ELSE 0 END) as acquired_count,
    SUM(CASE WHEN status = 'closed' THEN 1 ELSE 0 END) as closed_count
FROM startups
WHERE market != 'Unknown'
GROUP BY market
HAVING COUNT(*) >= 5
ORDER BY avg_efficiency_score DESC
LIMIT 20
"""
q2 = run_query(query=query2)
q2.to_csv('industry_breakdown.csv', index=False)

### Funding stage analysis 

Do startups that reach later funding rounds score higher ? 


In [337]:
query3 = """
SELECT
    CASE 
        WHEN round_D > 0 THEN '4. Series D+'
        WHEN round_C > 0 THEN '3. Series C'
        WHEN round_B > 0 THEN '2. Series B'
        WHEN round_A > 0 THEN '1. Series A'
        WHEN venture > 0 THEN '0. Venture Only'
        WHEN seed > 0 THEN '0. Seed Only'
        ELSE 'No Round Data'
    END as highest_round,
    COUNT(*) as company_count,
    ROUND(AVG(efficiency_score), 2) as avg_efficiency,
    ROUND(AVG(funding_total_usd), 0) as avg_funding_raised,
    SUM(CASE WHEN status = 'acquired' THEN 1 ELSE 0 END) as acquired_count
FROM startups
GROUP BY highest_round
HAVING highest_round != 'No Round Data'
ORDER BY highest_round"""

q3 = run_query(query=query3)
q3.to_csv('funding_stage_analysis.csv', index=False)

### Query 4: geographical analysis 

what cities and regions produce the most efficient startups 

In [338]:
query4 = """
SELECT 
    city,
    COUNT(*) as company_count,
    ROUND(AVG(efficiency_score), 2) as avg_efficiency,
    ROUND(AVG(funding_total_usd), 0) as avg_funding,
    SUM(CASE WHEN status = 'acquired' THEN 1 ELSE 0 END) as acquisitions
FROM startups
WHERE city != 'Unknown' 
GROUP BY city
HAVING COUNT(*) >= 20
ORDER BY avg_efficiency DESC
LIMIT 20
"""

q4 = run_query(query=query4)
q4.to_csv('eff_by_city.csv', index=False)

### Query 5 : efficiency vs funding 

do startups that raise more money actually score better 

In [339]:
query5 = """
WITH ranked AS (
    SELECT *, NTILE(5) OVER (ORDER BY funding_total_usd) as funding_quintile
    FROM startups
)
SELECT
    CASE funding_quintile
        WHEN 1 THEN '1. Bottom 20%'
        WHEN 2 THEN '2. Lower Mid 20%'
        WHEN 3 THEN '3. Middle 20%'
        WHEN 4 THEN '4. Upper Mid 20%'
        WHEN 5 THEN '5. Top 20%'
    END as funding_percentile,
    COUNT(*) as company_count,
    ROUND(AVG(efficiency_score), 2) as avg_efficiency
FROM ranked
GROUP BY funding_quintile
ORDER BY funding_quintile
"""

q5 = run_query(query=query5)
q5.to_csv('efficiency_corr.csv', index=False)